# Neural + Neighbors Pipeline

This notebook created with help from Clause


CPU-only notebook. The `*_neighbor_only.csv` files are already in `src/` (copied from milestone6).

**Steps:**
1. (Optional) Inspect and merge CSVs — saves `*_merged.csv` for inspection
2. Run the full pipeline — trains RoutedBlend + MLP stacking, writes `submission.csv`

**Changes from milestone6:**
- `max_features`: 80 → 100
- `n_neighbors`: 7 → 15 (if re-running feature extraction)


In [ ]:
!pip install -q pandas numpy scikit-learn xgboost torch

In [ ]:
import sys
from pathlib import Path

# ── Paths — adjust if running from a different location ───────────────────────
SRC_DIR  = Path(".").resolve()           # Neural_and_Neighbors/src/
DATA_ROOT = SRC_DIR / ".." / ".." / "milestone6" / "data" / "extracted_features"
OUTPUT_DIR = SRC_DIR / ".." / "output"

DATA_ROOT  = DATA_ROOT.resolve()
OUTPUT_DIR = OUTPUT_DIR.resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))

assert DATA_ROOT.exists(),  f"Data not found at {DATA_ROOT}"
assert (SRC_DIR / "train_neighbor_only.csv").exists(), "train_neighbor_only.csv missing from src/"

print(f"SRC_DIR   : {SRC_DIR}")
print(f"DATA_ROOT : {DATA_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"\nFiles in src/:")
print(sorted(f.name for f in SRC_DIR.iterdir() if not f.name.startswith("_")))

## (Optional) Step 1 — Create merged CSVs for inspection
Merges all 3 family CSVs with neighbor features and saves `*_merged.csv` here.
Skip this if you only want the submission.

In [ ]:
from create_merged_csvs import create_merged

for split in ["train", "val", "test"]:
    print(f"Merging {split}...")
    create_merged(DATA_ROOT, SRC_DIR, split)

## Step 2 — Run the full pipeline

In [ ]:
from pipeline_with_neural import run_pipeline

run_pipeline(DATA_ROOT, SRC_DIR, OUTPUT_DIR)

In [ ]:
import json

metrics = json.loads((OUTPUT_DIR / "metrics.json").read_text())
print(f"RoutedBlend AUC : {metrics['routed_blend_auc']:.4f}")
print(f"MLP Stacking AUC: {metrics['mlp_stacking_auc']:.4f}  TPR@FPR=0.1: {metrics['mlp_tpr_at_fpr01']:.4f}")
print(f"Best model      : {metrics['best_model']}  (AUC={metrics['best_val_auc']:.4f})")
print(f"Features used   : {metrics['routed_details']['feature_count']}")
print(f"\nTop features:")
for f in metrics['routed_details']['top_features']:
    print(f"  {f}")
print(f"\nSubmission → {OUTPUT_DIR / 'submission.csv'}")